# Fill Weight Analysis - Plotting Demo

This notebook demonstrates control charting with real manufacturing data using the ProcessBehavior library.

## Dataset: FILLWEIGHTDATA_800

- **800 observations** from a bottle filling process
- **100 time points** (pulls)
- **4 lanes** x **2 phases** = **8 Lane/Phase combinations**

## New API with Auto-Complete

The library now supports IDE auto-complete for column names:

```python
pdf = ProcessDataFrame(df)
study = pdf.formulate(
    factors=[pdf.columns.lane, pdf.columns.phase],
    time=pdf.columns.pull,
    response=pdf.columns.fill_weight
)
result = study.analyze('Xbar')  # or 'Imr', 'S', etc.
```

## Two Analysis Approaches

1. **Xbar-S Charts** - 8 points showing mean/std dev per Lane/Phase combination
2. **Stratified IMR** - 8 individual time series charts, one per Lane/Phase

In [1]:
import pandas as pd
from processbehavior import ProcessBehavior

# Load fill weight data
df = pd.read_csv('../processbehavior/datasets/data/FILLWEIGHTDATA_800.csv')

print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

Shape: (800, 4)
Columns: ['pull', 'lane', 'phase', 'fill_weight']


,pull,lane,phase,fill_weight
0,1,1,1,236.93
1,1,1,2,237.39
2,1,2,1,236.30
3,1,2,2,241.35
4,1,3,1,236.09


## Create ProcessDataFrame with Auto-Complete

The `ProcessDataFrame` wrapper provides `pdf.columns` for IDE auto-complete.

In [2]:
# Create ProcessBehavior
pb = ProcessBehavior(df)

# Auto-complete: type pb.cols. and your IDE will show available columns
print("Available columns via pb.cols:")
print(f"  pb.cols.lane       -> '{pb.cols.lane}'")
print(f"  pb.cols.phase      -> '{pb.cols.phase}'")
print(f"  pb.cols.pull       -> '{pb.cols.pull}'")
print(f"  pb.cols.fill_weight -> '{pb.cols.fill_weight}'")

Available columns via pb.cols:
  pb.cols.lane       -> 'lane'
  pb.cols.phase      -> 'phase'
  pb.cols.pull       -> 'pull'
  pb.cols.fill_weight -> 'fill_weight'


---
# Part 1: Xbar-S Charts

Create a Study using `formulate()`, then analyze with the desired chart type.

In [3]:
# Create a Study with formulate() - uses auto-complete for column names
study = pb.formulate(
    factors=[pb.cols.lane, pb.cols.phase],
    time=pb.cols.pull,
    response=pb.cols.fill_weight
)

# Execute with Xbar chart (produces Xbar and Sbar charts)
result_xbar = study.execute('Xbar')

print(result_xbar)

ANALYSIS RESULT SUMMARY

Sampling Design State: SDS 1
Description: Full replication (all cells n≥2)

Analysis Type: Xbar
Response Variable: fill_weight
Grouping: lane, phase
Time Variable: pull

Observations: 789
Charts: Xbar, Sbar

Capabilities:
  Residuals: ✓
  Effects: ✓
  Interactions: ✓

⚠️  Signals: 9 points beyond limits


In [4]:
# Available charts from this analysis
print("Available charts:")
for chart_name in result_xbar.charts.keys():
    n_points = len(result_xbar.charts[chart_name]['data'])
    print(f"  {chart_name}: {n_points} points")

Available charts:
  Xbar: 8 points
  Sbar: 8 points


In [5]:
# Plot Xbar chart (8 points - one per Lane/Phase combination)
fig = result_xbar.plot(
    chart='Xbar',
    show_zones=True,
    show_stats=True,
    show_rules=True,
    title='Xbar Chart - Mean Fill Weight by Lane/Phase'
)
fig.show()

In [6]:
# Plot Sbar chart (8 points - one per Lane/Phase combination)
fig = result_xbar.plot(
    chart='Sbar',
    show_zones=True,
    show_stats=True,
    show_rules=True,
    title='S Chart - Std Dev by Lane/Phase'
)
fig.show()

---
# Part 2: Stratified IMR Charts

Using the same Study, analyze with `'Imr'` to get 8 individual time series charts.

In [7]:
# Execute with IMR chart - same study, different chart type
result_imr = study.execute('Imr')

print(result_imr)

ANALYSIS RESULT SUMMARY

Sampling Design State: SDS 1
Description: Full replication (all cells n≥2)

Analysis Type: Xbar
Response Variable: fill_weight
Grouping: lane, phase
Time Variable: pull

Observations: 789
Charts: 1_1, 1_2, 2_1, 2_2, 3_1, 3_2, 4_1, 4_2
Stratified: Yes (8 groups)

Capabilities:
  Residuals: ✓
  Effects: ✓
  Interactions: ✓

⚠️  Signals: 66 points beyond limits


In [8]:
# Available charts (8 stratified IMR charts - one per Lane/Phase)
print("Available charts (Stratified IMR):")
for chart_name in result_imr.charts.keys():
    if chart_name != 'all':
        n_points = len(result_imr.charts[chart_name]['data'])
        print(f"  {chart_name}: {n_points} points")

Available charts (Stratified IMR):
  1_1: 99 points
  1_2: 98 points
  2_1: 100 points
  2_2: 98 points
  3_1: 99 points
  3_2: 98 points
  4_1: 99 points
  4_2: 98 points


In [9]:
# Plot all 8 stratified IMR charts as facets
fig = result_imr.plot(
    show_zones=True,
    show_rules=True,
    ncols=4,
    width=1400,
    height=700,
    title='Stratified IMR Charts - One per Lane/Phase Combination'
)
fig.show()

In [10]:
# Plot a single stratum in detail
# Chart names are "lane_phase" format
fig = result_imr.plot(
    chart='1_1',
    show_zones=True,
    show_stats=True,
    show_rules=True,
    title='IMR Chart - Lane 1, Phase 1'
)
fig.show()

---
## Signal Detection

Detect signals (out-of-control points) across all stratified charts.

In [11]:
# Signal detection for stratified IMR charts
signals = result_imr.detect_signals()

print("Signal Detection - Stratified IMR Charts")
print("=" * 50)
total = 0
for chart_name, signal_result in signals.items():
    if signal_result.has_signals:
        n = len(signal_result.violations)
        total += n
        rule_counts = signal_result.violations.groupby('rule_name').size()
        rules_str = ', '.join([f"{r}: {c}" for r, c in rule_counts.items()])
        print(f"{chart_name}: {n} violations ({rules_str})")
    else:
        print(f"{chart_name}: No signals")

print(f"\nTotal signals: {total}")

Signal Detection - Stratified IMR Charts
1_1: 18 violations (rule_1: 6, rule_3: 3, rule_4: 7, rule_7: 2)
1_2: 10 violations (rule_1: 3, rule_2: 1, rule_3: 2, rule_4: 4)
2_1: 54 violations (rule_1: 10, rule_2: 5, rule_3: 21, rule_4: 12, rule_8: 6)
2_2: 74 violations (rule_1: 13, rule_2: 6, rule_3: 24, rule_4: 24, rule_8: 7)
3_1: 82 violations (rule_1: 6, rule_2: 5, rule_3: 34, rule_4: 26, rule_5: 2, rule_8: 9)
3_2: 59 violations (rule_1: 4, rule_2: 5, rule_3: 21, rule_4: 29)
4_1: 146 violations (rule_1: 11, rule_2: 10, rule_3: 48, rule_4: 66, rule_8: 11)
4_2: 162 violations (rule_1: 13, rule_2: 13, rule_3: 55, rule_4: 56, rule_5: 1, rule_8: 24)

Total signals: 605


---
## Theme Options

The library supports light and dark themes.

In [12]:
# Dark theme
fig = result_imr.plot(
    theme='dark',
    show_zones=True,
    ncols=4,
    width=1400,
    height=700,
    title='Stratified IMR Charts - Dark Theme'
)
fig.show()

---
## Summary

### New API Pattern

```python
pdf = ProcessDataFrame(df)
study = pdf.formulate(
    factors=[pdf.columns.lane, pdf.columns.phase],
    time=pdf.columns.pull,
    response=pdf.columns.fill_weight
)

# Different chart types from the same study
result_xbar = study.analyze('Xbar')  # Xbar + Sbar charts
result_imr = study.analyze('Imr')    # Stratified IMR charts
```

### Chart Types

| Analysis | Method | Charts | Description |
|----------|--------|--------|-------------|
| **Xbar-S** | `study.analyze('Xbar')` | Xbar, Sbar | Compare subgroup means/variability |
| **Stratified IMR** | `study.analyze('Imr')` | One per stratum | Monitor each group over time |

### Key Features

- **Auto-complete**: `pdf.columns.<column_name>` for IDE support
- **Two-tier signals**: Red (beyond limits), Orange (run rule patterns)
- **Shared Y-Axis**: `shared_yaxis=True` (default) for honest comparison
- **Zone shading**: `show_zones=True` to visualize sigma zones
- **Themes**: `theme='light'` or `theme='dark'`

In [13]:
# Shared Y-Axis (default) - enables honest comparison across facets
fig = result_imr.plot(
    show_zones=True,
    show_rules=True,
    shared_yaxis=True,      # Same y-scale for all facets (default)
    yaxis_padding=0.05,     # 5% padding (default)
    ncols=4,
    width=1400,
    height=700,
    title='Shared Y-Axis - Same Scale for All Facets (Recommended)'
)
fig.show()

In [14]:
# Free Y-Axis - each facet scales independently
# Useful for seeing local patterns, but can be misleading for comparisons
fig = result_imr.plot(
    show_zones=True,
    show_rules=True,
    shared_yaxis=False,  # Each facet has its own scale
    ncols=4,
    width=1400,
    height=700,
    title='Free Y-Axis - Independent Scale per Facet'
)
fig.show()

---
## Summary

| Analysis | Code | Charts | Points per Chart |
|----------|------|--------|------------------|
| **Xbar-S** (default) | `grouping_vars=['lane', 'phase']` | 2 (Xbar, Sbar) | 8 |
| **Stratified IMR** | `grouping_vars=['lane', 'phase'], chart_type='Imr'` | 8 (one per stratum) | ~100 |

### Key Points

- **Xbar-S**: Compare Lane/Phase combinations against each other
- **Stratified IMR**: Monitor each Lane/Phase combination over time
- **Two-tier signals**: Red (Rule 1 - beyond limits), Orange (Rules 2-8 - patterns)